# Eventos ao vivo, audiência e QoE do dump Argus

## Escopo, identidade e limites de comparação

O EPG identifica um confronto por `event_key`; `tid` permanece visível como a
identidade do feed. Slots simultâneos não são fundidos pelo horário de início.
Cada leitura usa o intervalo EPG completo e uma janela de audiência/QoE de **15 minutos**,
com 60 minutos antes e depois apenas para contexto.

O slug `globo` é associado à família `globo%`, uma aproximação nacional marcada
como `national_family_approximation`. Outros slugs usam correspondência exata.
A ausência de telemetria para uma linha do EPG é exibida como
`telemetry_available=False`; especialmente para slots futuros ou cobertura QoE
atrasada, isso não é um incidente.

In [1]:
import pandas as pd

from dump_explorer.charts import event_audience_timeline, event_impact_scatter, qoe_timeline
from dump_explorer.config import load_settings
from dump_explorer.database import create_db_engine
from dump_explorer.live import (
    build_event_comparison,
    distinct_event_count,
    load_event_audience,
    load_live_events,
)

# derive a safe UTC epoch-ms window from signal_bucket observed bounds
from dump_explorer.quality import load_source_coverage

settings = load_settings()
engine = create_db_engine(settings)

# Obter janela segura a partir da cobertura de dados
s = load_source_coverage(engine)
if s.empty or s['first_ts'].dropna().empty:
    print("⚠️  source_coverage vazio. Usando valores padrão.")
    start_ms = 0
    end_ms = 86_400_000
else:
    start_ms = int(s['first_ts'].dropna().min().value // 10**6)
    end_ms = int(s['last_ts'].dropna().max().value // 10**6) + 1
    print(f"Intervalo de dados: {pd.to_datetime(start_ms, unit='ms', utc=True)} a {pd.to_datetime(end_ms-1, unit='ms', utc=True)}")

Intervalo de dados: 2026-07-10 18:45:00+00:00 a 2026-08-05 23:45:00+00:00


## 1. Catálogo EPG e confrontos distintos

A contagem usa `event_key`; `tid` mantém a identidade do feed.

In [2]:
epg_events = load_live_events(engine, start_ms, end_ms)
display(epg_events)
display({"distinct_confrontations": distinct_event_count(epg_events)})

,event_key,tid,slug,start_ts,end_ts
0,2026-07-10:belgica|espanha,994,globo,1783708500000,1783717799000
1,2026-07-10:belgica|espanha,995,globo,1783708500000,1783717799000
2,2026-07-15:argentina|inglaterra,994,globo,1784140200000,1784149799000
3,2026-07-15:argentina|inglaterra,995,globo,1784140200000,1784149799000
4,2026-07-18:franca|inglaterra,994,globo,1784407500000,1784416199000
5,2026-07-18:franca|inglaterra,995,globo,1784407500000,1784416199000
6,2026-07-19:argentina|espanha,994,globo,1784487600000,1784498699000
7,2026-07-19:argentina|espanha,995,globo,1784487600000,1784498699000
8,2026-07-22:athletico-pr|sao-paulo,995,globo,1784766000000,1784773799000
9,2026-07-22:chapecoense|flamengo,994,globo,1784766000000,1784773799000


{'distinct_confrontations': 12}

## 2. Audiência antes, durante e depois do evento

Escolha confrontos explicitamente. Os pontos são sessões ativas concorrentes,
não entradas acumuladas. O slot EPG recebe 60 minutos de contexto de cada lado.

In [3]:
event_keys = []  # selecione event_key(s) vistos no catálogo EPG

if not event_keys:
    print("⚠️  event_keys está vazia. Selecione event_key(s) do catálogo EPG acima para visualizar audiência.")
else:
    timelines = [load_event_audience(engine, key, padding_minutes=60) for key in event_keys]
    event_timeline = pd.concat(timelines, ignore_index=True) if timelines else pd.DataFrame()
    display(event_timeline)
    if not event_timeline.empty:
        try:
            event_audience_timeline(event_timeline).show()
        except Exception as e:
            print(f"❌ Erro ao renderizar timeline de audiência: {e}")
    else:
        print(f"⚠️  Nenhum dado de audiência encontrado para os event_keys: {event_keys}")

⚠️  event_keys está vazia. Selecione event_key(s) do catálogo EPG acima para visualizar audiência.


## 3. Erros, buffering e join time

`active_with_error` representa sessões ativas com erro; buffering é ratio e
`jointime_s_avg` é em segundos. Só trace pontos com `telemetry_available=True`;
não preencha lacunas com zero.

In [4]:
event_timeline = globals().get("event_timeline", pd.DataFrame())
if not event_timeline.empty and "telemetry_available" in event_timeline.columns:
    observed = event_timeline.loc[event_timeline["telemetry_available"] == True].copy()
    if not observed.empty:
        available_metrics = [
            metric
            for metric in ["active_with_error", "bufferratio_avg", "jointime_s_avg"]
            if metric in observed.columns
        ]
        if available_metrics:
            for metric in available_metrics:
                qoe_timeline(observed, metric=metric).show()
        else:
            print("A telemetria está disponível, mas as colunas de QoE esperadas não foram encontradas.")
    else:
        print("O EPG foi encontrado, mas não há telemetria disponível nesta janela.")
else:
    print("Não há evento carregado ou a coluna telemetry_available não existe nesta seleção.")

Não há evento carregado ou a coluna telemetry_available não existe nesta seleção.


## 4. Comparação entre confrontos

A comparação conserva feeds com EPG sem telemetria. O gráfico posiciona somente
confrontos observados e anota os demais; sobreposição temporal não prova causalidade.

In [5]:
if event_keys:
    try:
        comparison = build_event_comparison(engine, event_keys)
        display(comparison)
        if not comparison.empty:
            event_impact_scatter(comparison).show()
        else:
            print("⚠️  Nenhum dado de comparação encontrado para os event_keys especificados.")
    except Exception as e:
        print(f"❌ Erro ao construir comparação de eventos: {e}")
else:
    print("⚠️  Selecione event_key(s) para comparar audiência e QoE.")

⚠️  Selecione event_key(s) para comparar audiência e QoE.
